In [19]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HTTP_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["HTTPS_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["http_proxy"]= "http://proxy.utwente.nl:3128"
os.environ["https_proxy"]= "http://proxy.utwente.nl:3128"

Supervised

In [20]:
# constraints/load_allowed_labels.py
import json

def load_allowed_set(path="/home/simonettos/thijs/data_augmentatio_stefano/siamese_unsup_sup/allowed_labels_bosch.json") -> set[str]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    labels = data.get("labels", []) or []
    labels = {x.strip() for x in labels if isinstance(x, str) and x.strip()}
    return labels


In [21]:
"""
SUPERVISED BASELINE (straightforward classifier)
- Input: sentence only
- Output: multi-label logits over TTP IDs (NO TTP description used)
- Train: BCEWithLogitsLoss (+ optional pos_weight)
- Leading metric: hit@1 (also hit@5/10, mean_recall@k, MRR)

Assumes:
- TRAIN_DATA_PATH is a JSON list of {"sentence": str, "labels": [TTP,...]} or similar
- VALIDATION_DATA_PATH is your sentence-level validation split (list of dicts or (sent, labels))
- CONSTRAINT provides allowed labels like in your current pipeline (constraint_data["labels"] is a dict of id->list[ttp])
"""
from typing import Optional
import os
import json
import random
import math
import time
from dataclasses import dataclass
from typing import List, Tuple, Dict, Any

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm import tqdm


# -------------------------
# CONFIG
# -------------------------
TRAIN_DATA_PATH = "/home/simonettos/thijs/data_augmentatio_stefano/siamese_unsup_sup/train_clean.json"
VALIDATION_DATA_PATH = "/home/simonettos/thijs/data_augmentatio_stefano/siamese_unsup_sup/val_clean.json"

OUTPUT_DIR = "/home/simonettos/thijs/data_augmentatio_stefano/siamese_unsup_sup/sup_model_test_only"

BASE_MODEL = "ehsanaghaei/SecureBERT"

BATCH_SIZE = 16
PATIENCE   = 5
SEED       = 42

MAX_LEN_SENT = 192

LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_EPOCHS = 30

MAX_GRAD_NORM = 1.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Ranking metric configuration
K_LIST = (1, 5, 10)
TOPK_CAP = 200  # optional speed cap for metrics (set None to use all labels)


# -------------------------
# REPRODUCIBILITY
# -------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


# -------------------------
# HELPERS: data normalization
# -------------------------
def normalize_label_list(lbls):
    out = []
    for x in (lbls or []):
        if isinstance(x, str):
            x = x.strip()
            if x:
                out.append(x)
    return out

def iter_sentence_label_items(items):
    """Yields (sentence:str, labels:list[str]) from dict-items or tuple/list-items."""
    for it in items:
        if isinstance(it, dict):
            sent = (it.get("sentence") or "").strip()
            lbls = it.get("labels", []) or []
        else:
            sent = (it[0] or "").strip()
            lbls = it[1] if len(it) > 1 else []
        if not sent:
            continue
        lbls = normalize_label_list(lbls)
        if lbls:
            yield sent, lbls

def get_val_sentence(item):
    if isinstance(item, dict):
        return (item.get("sentence") or "").strip()
    if isinstance(item, (list, tuple)):
        return (item[0] or "").strip()
    raise TypeError(f"Unsupported val item type: {type(item)}")

def get_val_labels(item):
    if isinstance(item, dict):
        return item.get("labels", []) or []
    if isinstance(item, (list, tuple)):
        return item[1] if len(item) > 1 else []
    raise TypeError(f"Unsupported val item type: {type(item)}")


# -------------------------
# LOAD DATA
# -------------------------
with open(TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(VALIDATION_DATA_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)

def collect_labels(items):
    s = set()
    for _, lbls in iter_sentence_label_items(items):
        for l in lbls:
            s.add(l)
    return s

labels_in_train = collect_labels(train_data)
labels_in_val   = collect_labels(val_data)
labels_in_data  = labels_in_train | labels_in_val

allowed_set = load_allowed_set()
allowed_set = set(x for x in allowed_set if x in labels_in_data)

print("Allowed labels after intersect with data:", len(allowed_set))
print("Labels present in train:", len(labels_in_train))
print("Labels present in val:", len(labels_in_val))



# -------------------------
# LABEL INDEXING (classifier output space)
# -------------------------
label_list = sorted(list(allowed_set))  # stable order
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

print("NUM_LABELS:", NUM_LABELS)
print("NUM_LABELS (scored per sentence):", NUM_LABELS)


# -------------------------
# DATASET + COLLATE
# -------------------------
class MultiLabelDataset(Dataset):
    """
    Produces:
      - sentence (str)
      - y (float tensor [NUM_LABELS]) multi-hot
    """
    def __init__(self, items):
        samples = {}
        # merge duplicates at sentence level (union labels) to reduce noise
        for sent, lbls in iter_sentence_label_items(items):
            lbls_f = [l for l in lbls if l in allowed_set]
            if not lbls_f:
                continue
            if sent not in samples:
                samples[sent] = set()
            samples[sent].update(lbls_f)
        self.samples = [(s, sorted(v)) for s, v in samples.items() if v]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sent, lbls = self.samples[idx]
        y = torch.zeros(NUM_LABELS, dtype=torch.float32)
        for l in lbls:
            y[label2id[l]] = 1.0
        return sent, y

@dataclass
class SentCollator:
    tokenizer: Any
    max_len: int

    def __call__(self, batch):
        sents = [b[0] for b in batch]
        ys    = torch.stack([b[1] for b in batch], dim=0)

        tok = self.tokenizer(
            sents,
            padding=True,
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        return tok, ys


# -------------------------
# MODEL: SecureBERT encoder + linear multi-label head
# -------------------------
class SecureBertMultiLabel(nn.Module):
    def __init__(self, model_name: str, num_labels: int):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.classifier = nn.Linear(hidden, num_labels)

    @staticmethod
    def mean_pool(last_hidden: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
        summed = (last_hidden * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-9)
        return summed / denom

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        emb = self.mean_pool(out.last_hidden_state, attention_mask)
        logits = self.classifier(emb)  # [B, NUM_LABELS]
        return logits


# -------------------------
# pos_weight for imbalance (recommended)
# -------------------------
def compute_pos_weight(train_ds: MultiLabelDataset) -> torch.Tensor:
    pos = torch.zeros(NUM_LABELS, dtype=torch.float32)
    for _, y in train_ds:
        pos += y
    total = float(len(train_ds))
    neg = total - pos
    # (neg/pos) with caps to avoid crazy gradients
    pos_weight = neg / (pos + 1e-6)
    pos_weight = torch.clamp(pos_weight, min=1.0, max=50.0)
    return pos_weight


# -------------------------
# VALIDATION: hit@k, mean_recall@k, MRR (ranking on logits)
# -------------------------
@torch.no_grad()
def validate_ranking(
    model: SecureBertMultiLabel,
    tokenizer,
    val_items,
    k_list=(1, 5, 10),
    max_len=192,
    topk_cap: Optional[int] = None,
    batch_size: int = 32,
) -> Dict[str, float]:
    model.eval()

    hits_at_k = {k: 0 for k in k_list}
    mean_recall_at_k = {k: 0.0 for k in k_list}
    mrr = 0.0
    used = 0

    # batch sentences for speed
    sentences = []
    golds = []

    for item in val_items:
        sent = get_val_sentence(item)
        raw_lbls = get_val_labels(item)
        gold = {t.strip() for t in raw_lbls if isinstance(t, str) and t.strip()}
        gold = {t for t in gold if t in label2id}
        if not sent or not gold:
            continue
        sentences.append(sent)
        golds.append(gold)

    if not sentences:
        return {**{f"hit@{k}": 0.0 for k in k_list},
                **{f"mean_recall@{k}": 0.0 for k in k_list},
                "mrr": 0.0,
                "val_used": 0}

    for i in range(0, len(sentences), batch_size):
        batch_sents = sentences[i:i+batch_size]
        batch_golds = golds[i:i+batch_size]

        tok = tokenizer(
            batch_sents,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        )
        tok = {k: v.to(DEVICE) for k, v in tok.items()}

        logits = model(**tok)  # [B, L]
        scores = torch.sigmoid(logits)  # optional; ranking is same as logits monotonic transform

        # optional cap: only consider the top N predictions for MRR loop speed
        if topk_cap is None:
            topk_cap = scores.size(1)
        cap = min(int(topk_cap), scores.size(1))

        # top cap indices per row
        top_scores, top_idx = torch.topk(scores, k=cap, dim=1, largest=True, sorted=True)

        for row in range(scores.size(0)):
            gold = batch_golds[row]
            used += 1

            # compute MRR within top cap (if not found, RR=0)
            rr = 0.0
            for rank_pos in range(cap):
                pred_id = id2label[int(top_idx[row, rank_pos].item())]
                if pred_id in gold:
                    rr = 1.0 / float(rank_pos + 1)
                    break
            mrr += rr

            # compute hit@k and mean_recall@k using the same ranking (full or top cap)
            for k in k_list:
                kk = min(k, cap)
                topk = [id2label[int(x)] for x in top_idx[row, :kk].tolist()]
                hits_at_k[k] += int(any(t in gold for t in topk))
                mean_recall_at_k[k] += len(set(topk) & gold) / max(1, len(gold))

    n = max(1, used)
    metrics = {f"hit@{k}": hits_at_k[k] / n for k in k_list}
    metrics.update({f"mean_recall@{k}": mean_recall_at_k[k] / n for k in k_list})
    metrics["mrr"] = mrr / n
    metrics["val_used"] = used
    return metrics


# -------------------------
# SAVE / LOAD
# -------------------------
def save_model(model: SecureBertMultiLabel, tokenizer, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    model.backbone.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    torch.save(model.classifier.state_dict(), os.path.join(out_dir, "classifier.pt"))

def load_model(out_dir: str, base_model: str, num_labels: int):
    tokenizer = AutoTokenizer.from_pretrained(out_dir, use_fast=True)
    model = SecureBertMultiLabel(base_model, num_labels)
    model.backbone = AutoModel.from_pretrained(out_dir)
    cls_path = os.path.join(out_dir, "classifier.pt")
    model.classifier.load_state_dict(torch.load(cls_path, map_location="cpu"))
    return model, tokenizer


# -------------------------
# TOKENIZER / DATA / LOADER
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

train_ds = MultiLabelDataset(train_data)
val_ds   = MultiLabelDataset(val_data)

print("Train unique sentences:", len(train_ds))
print("Val   unique sentences:", len(val_ds))

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    num_workers=2,
    pin_memory=(DEVICE == "cuda"),
    collate_fn=SentCollator(tokenizer, MAX_LEN_SENT),
)

# We will validate directly from `val_data` to keep identical behavior to your other methods
# (same raw val items -> same gold extraction).
# If you prefer sentence-deduped validation, pass `val_ds.samples` converted to dicts/tuples.


# -------------------------
# INIT MODEL / OPT / SCHED
# -------------------------
model = SecureBertMultiLabel(BASE_MODEL, NUM_LABELS).to(DEVICE)

pos_weight = compute_pos_weight(train_ds).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)


# -------------------------
# TRAIN LOOP (early stop on hit@1)
# -------------------------
best_hit1 = -1.0
patience_ctr = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running = 0.0

    print(f"\nEpoch {epoch}/{MAX_EPOCHS}")
    for tok, labels in tqdm(train_loader, desc="Training", leave=False):
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(**tok)
        loss = criterion(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()

        running += loss.item()

    avg_loss = running / max(1, len(train_loader))

    metrics = validate_ranking(
        model=model,
        tokenizer=tokenizer,
        val_items=val_data,
        k_list=K_LIST,
        max_len=MAX_LEN_SENT,
        topk_cap=TOPK_CAP,
        batch_size=32,
    )

    print(f"Train loss: {avg_loss:.4f}")
    print("Validation:", ", ".join([f"{k}: {v:.4f}" for k, v in metrics.items()]))

    hit1 = metrics.get("hit@1", 0.0)

    if hit1 > best_hit1:
        best_hit1 = hit1
        patience_ctr = 0
        save_model(model, tokenizer, OUTPUT_DIR)
        print("✔ New best model saved (by hit@1)")
    else:
        patience_ctr += 1
        print(f"No improvement (patience {patience_ctr}/{PATIENCE})")

    if patience_ctr >= PATIENCE:
        print("Early stopping triggered")
        break

print(f"\nBest validation hit@1: {best_hit1:.4f}")
print(f"Saved to: {OUTPUT_DIR}")





model2, tok2 = load_model(OUTPUT_DIR, BASE_MODEL, NUM_LABELS)
model2 = model2.to(DEVICE)
metrics = validate_ranking(model2, tok2, val_data, k_list=(1,5,10), max_len=MAX_LEN_SENT, topk_cap=None)
print(metrics)



Allowed labels after intersect with data: 112
Labels present in train: 801
Labels present in val: 470
NUM_LABELS: 112
NUM_LABELS (scored per sentence): 112
Train unique sentences: 8926
Val   unique sentences: 1134


Some weights of RobertaModel were not initialized from the model checkpoint at ehsanaghaei/SecureBERT and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Epoch 1/30


Train loss: 0.7115
Validation: hit@1: 0.5062, hit@5: 0.7575, hit@10: 0.8254, mean_recall@1: 0.4832, mean_recall@5: 0.7442, mean_recall@10: 0.8156, mrr: 0.6192, val_used: 1134.0000
✔ New best model saved (by hit@1)

Epoch 2/30


Train loss: 0.3963
Validation: hit@1: 0.6755, hit@5: 0.9004, hit@10: 0.9330, mean_recall@1: 0.6499, mean_recall@5: 0.8917, mean_recall@10: 0.9251, mrr: 0.7735, val_used: 1134.0000
✔ New best model saved (by hit@1)

Epoch 3/30


...

Train loss: 0.0027
Validation: hit@1: 0.8245, hit@5: 0.9418, hit@10: 0.9630, mean_recall@1: 0.7966, mean_recall@5: 0.9345, mean_recall@10: 0.9578, mrr: 0.8775, val_used: 1134.0000
No improvement (patience 3/5)

Epoch 29/30


Train loss: 0.0026
Validation: hit@1: 0.8245, hit@5: 0.9383, hit@10: 0.9612, mean_recall@1: 0.7966, mean_recall@5: 0.9306, mean_recall@10: 0.9556, mrr: 0.8763, val_used: 1134.0000
No improvement (patience 4/5)

Epoch 30/30


Train loss: 0.0025
Validation: hit@1: 0.8263, hit@5: 0.9392, hit@10: 0.9612, mean_recall@1: 0.7983, mean_recall@5: 0.9314, mean_recall@10: 0.9562, mrr: 0.8773, val_used: 1134.0000
No improvement (patience 5/5)
Early stopping triggered

Best validation hit@1: 0.8280
Saved to: /home/simonettos/thijs/data_augmentatio_stefano/siamese_unsup_sup/sup_model_tram_test_only


Some weights of RobertaModel were not initialized from the model checkpoint at ehsanaghaei/SecureBERT and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'hit@1': 0.828042328042328, 'hit@5': 0.9409171075837742, 'hit@10': 0.9629629629629629, 'mean_recall@1': 0.7999338624338627, 'mean_recall@5': 0.9324294532627866, 'mean_recall@10': 0.9579291593180481, 'mrr': 0.8791489813443321, 'val_used': 1134}


In [24]:
TEST_DATA_PATH = "/home/simonettos/thijs/data_augmentatio_stefano/siamese_unsup_sup/test_clean.json"

with open(TEST_DATA_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)
model2, tok2 = load_model(OUTPUT_DIR, BASE_MODEL, NUM_LABELS)
model2 = model2.to(DEVICE)
metrics = validate_ranking(model2, tok2, test_data, k_list=(1,5,10), max_len=MAX_LEN_SENT, topk_cap=None)
print(metrics)

Some weights of RobertaModel were not initialized from the model checkpoint at ehsanaghaei/SecureBERT and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'hit@1': 0.8221814848762603, 'hit@5': 0.9450045829514208, 'hit@10': 0.9651695692025665, 'mean_recall@1': 0.7964252978918426, 'mean_recall@5': 0.9383440268866483, 'mean_recall@10': 0.9608310418576229, 'mrr': 0.8776158155468189, 'val_used': 1091}


In [23]:
import json, random, hashlib

SEED = 42
random.seed(SEED)

with open('/home/simonettos/thijs/data_augmentatio_stefano/mitre/only_sentence_to_ttp.json', "r", encoding="utf-8") as f:
    all_items = json.load(f)

# dedupe by exact sentence (union labels)
from collections import defaultdict
sent2labels = defaultdict(set)
for it in all_items:
    sent = (it.get("sentence") or "").strip()
    if not sent: 
        continue
    for l in (it.get("labels") or []):
        if isinstance(l, str) and l.strip():
            sent2labels[sent].add(l.strip())

dedup = [{"sentence": s, "labels": sorted(list(lbls))} for s, lbls in sent2labels.items()]
random.shuffle(dedup)

val_frac = 0.10
test_frac = 0.10
n_val = int(len(dedup) * val_frac)
n_test = int(len(dedup) * test_frac)
val_clean = dedup[:n_val]
test_clean = dedup[n_val:n_val+n_test]
train_clean = dedup[n_val+n_test:]

# sanity: no overlap
train_s = {x["sentence"] for x in train_clean}
val_s   = {x["sentence"] for x in val_clean}
test_s  = {x["sentence"] for x in test_clean}
print("Overlap after split:", len(train_s & val_s), len(train_s & test_s), len(val_s & test_s))

with open("/home/simonettos/thijs/data_augmentatio_stefano/siamese_unsup_sup/train_clean_tram.json", "w", encoding="utf-8") as f:
    json.dump(train_clean, f, indent=2)
with open("/home/simonettos/thijs/data_augmentatio_stefano/siamese_unsup_sup/val_clean_tram.json", "w", encoding="utf-8") as f:
    json.dump(val_clean, f, indent=2)
with open("/home/simonettos/thijs/data_augmentatio_stefano/siamese_unsup_sup/test_clean_tram.json", "w", encoding="utf-8") as f:
    json.dump(test_clean, f, indent=2)


Overlap after split: 0 0 0


In [24]:
def sentence_set(items):
    s = set()
    for it in items:
        sent = get_val_sentence(it)
        if sent:
            s.add(sent.strip())
    return s

train_sents = sentence_set(train_data)
val_sents   = sentence_set(val_data)

overlap = train_sents & val_sents
print("Train sentences:", len(train_sents))
print("Val sentences:", len(val_sents))
print("Exact sentence overlap:", len(overlap))
if overlap:
    print("Example overlap:", next(iter(overlap))[:200])


Train sentences: 21726
Val sentences: 2414
Exact sentence overlap: 0
